# 070 CASE 040 — Grazing: Munkarp

> **Experimental.** Built in the Swedish Space Data Lab (SDL 3.0) with
> [IMINT Engine](https://github.com/DigitalEarthSwedenExperimentalLab/imintengine). Not a validated product or service, and without support.

**Question:** Are pasture blocks in the agricultural block register grazed during the season?

**Area:** Munkarp, Höör municipality (Skåne), growing season 2025.

**Partners:** Swedish Board of Agriculture (Jordbruksverket), RISE

**Data:** Copernicus Sentinel-2 L2A from [Digital Earth Sweden](https://digitalearth.se),
fetched through its openEO API.

**Licence:** The notebook is Apache-2.0 like the repository's code; its saved outputs are CC0 1.0 (see [License](https://github.com/DigitalEarthSwedenExperimentalLab/imintengine#license)). Contains modified Copernicus Sentinel data 2025. LPIS blocks from Jordbruksverket's open WFS; the block outlines on the saved map keep Jordbruksverket's terms rather than CC0. Grazing model (pib-ml-grazing) MIT, RISE.

## Before you run

Install IMINT Engine at the commit these results were produced with, plus Jupyter and the plotting
libraries:

```bash
pip install "imint-engine[fetch,inference] @ git+https://github.com/DigitalEarthSwedenExperimentalLab/imintengine@b183e1abd4653cb843a75557b05c09915fb85904" jupyterlab folium matplotlib
```

The notebook reads Sentinel-2 from Digital Earth Sweden, so each user logs in with their own DES
account in the *Log in* cell below, either with EGI in the browser or with a DES username and
password; see [digitalearth.se](https://digitalearth.se) for access. Every data fetch goes through
that login, nothing is saved in the notebook, and without a login the fetch stops with an error.
There is no synthetic fallback: every figure comes from real Sentinel-2 data. A run takes about 15 minutes per block (a Sentinel-2 time series for the whole season, fetched from DES in 5-day chunks), so about 45 minutes for the three blocks below.

## Log in to Digital Earth Sweden

In [ ]:
from getpass import getpass

import imint.fetch
import openeo
from IPython.display import clear_output

# Log in with your own Digital Earth Sweden account. Choose one:
#   "egi"      - your EGI account in the browser: open the link shown here and enter the code
#   "password" - your DES username and password, typed when asked
LOGIN_METHOD = "egi"

des = openeo.connect("https://openeo.digitalearth.se")
if LOGIN_METHOD == "password":
    user, password = input("DES username: "), getpass("DES password: ")
    if not (user and password):
        raise ValueError("enter both a DES username and a password")
    des.authenticate_basic(user, password)
    del password
else:
    des.authenticate_oidc(provider_id="egi")

# Every IMINT Engine fetch below uses this login and no other credentials on the machine
# (the pinned engine has no public parameter for a connection yet).
imint.fetch._connect = lambda token=None, token_path=None: des
clear_output()  # keep the typed username and the login link out of a saved notebook
print("Logged in to Digital Earth Sweden.")


## 1. Method

```
AOI → fetch_lpis_polygons (Jordbruksverket WFS, open)
    → fetch_grazing_timeseries (DES Sentinel-2, 12 bands × dates through the season)
    → GrazingAnalyzer.predict_batch (CNN-biLSTM) → class and confidence per block
```

The register (LPIS) divides farmland into blocks with a land-use type; this notebook takes the
blocks of type pasture ("Bete"). `GrazingAnalyzer` is a standalone class: it reads a time series per
block rather than a single scene.

## 2. Pasture blocks from LPIS

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import folium
from imint.fetch import fetch_lpis_polygons, fetch_grazing_timeseries
from imint.analyzers.grazing import GrazingAnalyzer

AOI = {"west": 13.42, "south": 55.935, "east": 13.48, "north": 55.965}  # Munkarp
YEAR = 2025
N_BLOCKS = 3  # about 15 minutes each; raise to cover more of the area

lpis = fetch_lpis_polygons(AOI, agoslag="Bete", max_features=500)
print(f"LPIS pasture blocks: {len(lpis)}, total {lpis.geometry.area.sum() / 1e4:.0f} ha, CRS {lpis.crs}")

  Fetching LPIS polygons from Jordbruksverket WFS …


  LPIS: 254 block fetched
  LPIS: 80 block after ägoslag filter (['Bete'])
LPIS pasture blocks: 80, total 162 ha, CRS EPSG:3006


## 3. Time series and classification

In [3]:
import contextlib
import io
import time

id_col = "blockid" if "blockid" in lpis.columns else None
subset = lpis.head(N_BLOCKS).to_crs("EPSG:4326")  # the first blocks the WFS returns, not a sample
series = []
for i in range(len(subset)):
    if LOGIN_METHOD == "egi":
        # Renew the login between blocks, one renewal at a time: a block takes well under the
        # access token's lifetime, and the EGI login server rejects a refresh token used twice.
        des.authenticate_oidc(provider_id="egi")
    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()):  # the fetch reports every 5-day chunk
        part = fetch_grazing_timeseries(polygons=subset.iloc[[i]], year=YEAR, polygon_id_col=id_col,
                                        source="des", n_workers=6)
    series += part
    for s in part:
        if len(s.dates) < 10:  # the fetch skips chunks that fail; never classify a truncated season
            raise RuntimeError(f"block {s.polygon_id}: only {len(s.dates)} dates arrived; run the cell again")
        print(f"block {s.polygon_id}: {len(s.dates)} cloud-free dates, "
              f"{s.dates[0] if s.dates else '-'} … {s.dates[-1] if s.dates else '-'}, "
              f"{(time.time() - t0) / 60:.0f} min", flush=True)

with contextlib.redirect_stdout(io.StringIO()):  # the model reports where its weights are stored
    predictions = GrazingAnalyzer().predict_batch(series)
labels = {1: "grazed", 0: "no activity", -1: "error"}
for p in predictions:
    print(f"  {str(p.polygon_id):>12s}  {labels[p.predicted_class]:12s} confidence {p.confidence:.2f}")
counts = {v: sum(p.predicted_class == k for p in predictions) for k, v in labels.items()}
print(counts)

Authenticated using refresh token.


block 62033533019: 32 cloud-free dates, 2025-04-03 … 2025-10-18, 15 min


Authenticated using refresh token.


block 62053530837: 37 cloud-free dates, 2025-04-03 … 2025-10-18, 13 min


Authenticated using refresh token.


block 62053524728: 36 cloud-free dates, 2025-04-03 … 2025-10-18, 17 min


   62033533019  grazed       confidence 0.85
   62053530837  grazed       confidence 0.88
   62053524728  grazed       confidence 0.91
{'grazed': 3, 'no activity': 0, 'error': 0}


## 4. Results

The map is interactive when the notebook runs in Jupyter; GitHub's viewer shows only a placeholder for it.

In [4]:
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=14, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics", name="Satellite basemap",
).add_to(m)
colour = {1: "#2ca02c", 0: "#d62728", -1: "#ff7f0e"}
cls = {p.polygon_id: p.predicted_class for p in predictions}
for _, row in lpis.to_crs("EPSG:4326").iterrows():
    pid = row.get(id_col) if id_col else None
    c = colour.get(cls.get(pid), "#888888")
    folium.GeoJson(row.geometry.__geo_interface__,
                   style_function=lambda _, c=c: {"fillColor": c, "color": c, "weight": 1.5, "fillOpacity": 0.4},
                   popup=f"{pid}: {labels.get(cls.get(pid), 'not classified')}").add_to(m)
folium.LayerControl().add_to(m)
m

## 5. Interpretation

This run classifies the first three of the area's pasture blocks that the WFS returns. That is a
demonstration, not a sample, and there is no accuracy assessment here. The classification is
decision support for the Board of Agriculture's checks, not an automated decision. The number of
cloud-free dates differs between blocks; the model accepts series of different length. Early
mowing, drought or flooding can look like "no activity". Grey blocks on the map were not classified.

**Links:** [`imint/fetch.py`](https://github.com/DigitalEarthSwedenExperimentalLab/imintengine/blob/b183e1a/imint/fetch.py) ·
[`imint/analyzers/grazing.py`](https://github.com/DigitalEarthSwedenExperimentalLab/imintengine/blob/b183e1a/imint/analyzers/grazing.py)

## Provenance

The saved outputs were produced by running this notebook top to bottom against IMINT Engine commit [`b183e1a`](https://github.com/DigitalEarthSwedenExperimentalLab/imintengine/commit/b183e1abd4653cb843a75557b05c09915fb85904) with data from Digital Earth Sweden. Dates are acquisition dates, chosen among the scenes DES holds by the cloud fraction over the area in the scene classification (SCL).